# LNL-Ti: XGBoost Light + v2 hybrid

A separate accuracy-first experiment combining the clean XGBoost-Light control flow
with v2's GTSRB-specific augmentation and mild Mixup.

Training phases:
1. Epochs 1-30: targeted augmentation + Mixup.
2. Epochs 31-40: targeted augmentation, Mixup off, learning rate reduced 10x.
3. Epochs 41-60: clean-image fine-tuning at low learning rate.

Deliberate omissions: EMA, MoEx, RandAugment, hard-sample weights, and target-streak
stopping. Validation uses exact correct counts; the official test is evaluated once.

In [ ]:
from pathlib import Path
import os
import subprocess

# Pipeline 1/7 - Locate the repository without changing either source notebook.
REPOSITORY_URL = "https://github.com/Omid-Nejati/Locality-iN-Locality.git"
repository_root = next(
    (
        path
        for path in (
            Path.cwd(),
            Path.cwd() / "Locality-iN-Locality-main",
            Path.cwd() / "Locality-iN-Locality",
        )
        if (path / "LNL_MoEx.py").exists()
    ),
    None,
)

if repository_root is None:
    repository_root = Path.cwd() / "Locality-iN-Locality"
    subprocess.run(["git", "clone", REPOSITORY_URL, str(repository_root)], check=True)

os.chdir(repository_root)
print("Working directory:", Path.cwd())

In [ ]:
%pip install -q timm==0.9.16 einops scikit-learn tqdm

In [ ]:
import csv
import math
import random
import shutil
import zipfile
from dataclasses import dataclass
from urllib.request import urlretrieve

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

SEED = 42


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", device)

## Data

In [ ]:
# Pipeline 2/7 - Download, extract, and arrange the official GTSRB split.
DATA_DIR = Path("data")
ARCHIVE_BASE_URL = (
    "https://sid.erda.dk/public/archives/"
    "daaeac0d7ce1152aea9b61d9f1e19370"
)
ARCHIVES = (
    "GTSRB_Final_Training_Images.zip",
    "GTSRB_Final_Test_Images.zip",
    "GTSRB_Final_Test_GT.zip",
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
required_paths = (
    DATA_DIR / "GTSRB" / "Final_Training" / "Images",
    DATA_DIR / "GTSRB" / "Final_Test" / "Images",
    DATA_DIR / "GT-final_test.csv",
)

if not all(path.exists() for path in required_paths):
    for filename in ARCHIVES:
        destination = DATA_DIR / filename
        if not destination.exists():
            urlretrieve(f"{ARCHIVE_BASE_URL}/{filename}", destination)
        with zipfile.ZipFile(destination) as archive:
            archive.extractall(DATA_DIR)

test_images_dir = DATA_DIR / "GTSRB" / "Final_Test" / "Images"
test_root = DATA_DIR / "GTSRB" / "test"
test_root.mkdir(parents=True, exist_ok=True)

with (DATA_DIR / "GT-final_test.csv").open(newline="") as csv_file:
    for row in csv.DictReader(csv_file, delimiter=";"):
        class_dir = test_root / f"{int(row['ClassId']):04d}"
        destination = class_dir / row["Filename"]
        if not destination.exists():
            class_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(test_images_dir / row["Filename"], destination)

print("Dataset root:", (DATA_DIR / "GTSRB").resolve())

In [ ]:
# Pipeline 3/7 - v2's sign-aware augmentation; no RandAugment is used.
class ResizeWithPadding:
    def __init__(self, size=224, fill=128):
        self.size = size
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        scale = self.size / max(width, height)
        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))
        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )
        horizontal = self.size - new_width
        vertical = self.size - new_height
        return TF.pad(
            image,
            [horizontal // 2, vertical // 2,
             horizontal - horizontal // 2, vertical - vertical // 2],
            fill=self.fill,
        )


normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)
eval_transform = transforms.Compose(
    [ResizeWithPadding(224), transforms.ToTensor(), normalize]
)
targeted_train_transform = transforms.Compose(
    [
        ResizeWithPadding(224),
        transforms.RandomApply(
            [
                transforms.RandomAffine(
                    degrees=10,
                    translate=(0.05, 0.05),
                    scale=(0.90, 1.10),
                    shear=3,
                )
            ],
            p=0.70,
        ),
        transforms.RandomPerspective(distortion_scale=0.15, p=0.20),
        transforms.RandomApply(
            [
                transforms.ColorJitter(
                    brightness=0.25,
                    contrast=0.25,
                    saturation=0.20,
                    hue=0.02,
                )
            ],
            p=0.60,
        ),
        transforms.ToTensor(),
        normalize,
    ]
)

In [ ]:
# Pipeline 4/7 - Preserve complete GTSRB tracks in a deterministic validation split.
TRAIN_ROOT = DATA_DIR / "GTSRB" / "Final_Training" / "Images"
TEST_ROOT = DATA_DIR / "GTSRB" / "test"
TARGET_VALIDATION_SIZE = 4000
MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64

# Phase changes mutate the dataset transform, so workers stay in the main process.
NUM_WORKERS = 0

index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
indices = np.arange(len(index_dataset))
targets = np.asarray(index_dataset.targets)


def track_id(sample_path):
    path = Path(sample_path)
    parts = path.stem.split("_")
    if len(parts) != 2 or not all(part.isdigit() for part in parts):
        raise ValueError(
            f"Unexpected GTSRB filename {path.name!r}; "
            "expected <TrackID>_<FrameID>.ppm."
        )
    return f"{path.parent.name}:{parts[0]}"


groups = np.asarray([track_id(path) for path, _ in index_dataset.samples])
group_counts = [
    len(set(groups[targets == class_id]))
    for class_id in range(len(index_dataset.classes))
]
nominal_splits = max(2, round(len(index_dataset) / TARGET_VALIDATION_SIZE))
max_splits = min(group_counts)
candidate_splits = range(
    max(2, nominal_splits - 2),
    min(max_splits, nominal_splits + 2) + 1,
)

split_candidates = []
for n_splits in candidate_splits:
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED,
    )
    for train_indices, validation_indices in splitter.split(
        indices,
        targets,
        groups,
    ):
        distance = abs(len(validation_indices) - TARGET_VALIDATION_SIZE)
        split_candidates.append((distance, train_indices, validation_indices))

if not split_candidates:
    raise RuntimeError("Could not build a track-safe validation split.")

_, train_indices, validation_indices = min(split_candidates, key=lambda item: item[0])
if set(groups[train_indices]) & set(groups[validation_indices]):
    raise RuntimeError("Track leakage detected between training and validation.")

training_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=targeted_train_transform,
)
validation_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=eval_transform,
)
test_dataset = torchvision.datasets.ImageFolder(TEST_ROOT, transform=eval_transform)

loader_generator = torch.Generator().manual_seed(SEED)
loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": device.type == "cuda",
}
training_loader = DataLoader(
    Subset(training_dataset, train_indices),
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    **loader_options,
)
validation_loader = DataLoader(
    Subset(validation_dataset, validation_indices),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    **loader_options,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    **loader_options,
)

print(
    f"Train: {len(train_indices):,} | "
    f"Validation: {len(validation_indices):,} | "
    f"Test: {len(test_dataset):,}"
)

## Train LNL-Ti hybrid

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti

# Hybrid schedule - v2 regularization first, XGBoost-style shrinkage to finish.
NUM_CLASSES = 43
NUM_EPOCHS = 60
MIN_EPOCHS = 45
EARLY_STOP_PATIENCE = 12
TARGET_TOP1 = 99.5

MIXUP_STOP_EPOCH = 30
CLEAN_START_EPOCH = 40
MIXUP_PROBABILITY = 0.35
MIXUP_ALPHA = 0.25

WARMUP_EPOCHS = 5
ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-4
COOLDOWN_LR_FACTOR = 0.10
MIN_LR_FACTOR = 0.01
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.02
GRAD_CLIP_NORM = 1.0
DROP_PATH_RATE = 0.10

default_checkpoint_dir = (
    Path("/content/drive/MyDrive/LNL_MoEx_checkpoints")
    if Path("/content/drive/MyDrive").exists()
    else Path("checkpoints")
)
CHECKPOINT_DIR = Path(
    os.environ.get("CHECKPOINT_DIR", str(default_checkpoint_dir))
)
CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "LNL_Ti_targeted_mixup_shrinkage_seed42_v7.pth"
)
RESUME_TRAINING = os.environ.get("RESUME_TRAINING", "1") == "1"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def build_model():
    return LNL_MoEx_Ti(
        pretrained=False,
        num_classes=NUM_CLASSES,
        drop_path_rate=DROP_PATH_RATE,
    ).to(device)


model = build_model()
decay_parameters = []
no_decay_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    if parameter.ndim <= 1 or name.endswith(".bias"):
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

optimizer = optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)


def learning_rate_factor(epoch):
    if epoch < WARMUP_EPOCHS:
        factor = (epoch + 1) / WARMUP_EPOCHS
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(
            1,
            NUM_EPOCHS - WARMUP_EPOCHS,
        )
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        factor = MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine

    # XGBoost analogy: shrink updates once Mixup is removed.
    if epoch >= MIXUP_STOP_EPOCH:
        factor *= COOLDOWN_LR_FACTOR
    return factor


scheduler = optim.lr_scheduler.LambdaLR(optimizer, learning_rate_factor)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
@dataclass
class TrainState:
    epoch: int = 0
    best_correct: int = -1
    best_top1: float = -1.0
    best_weights: dict = None
    stale_epochs: int = 0
    post_regularization_started: bool = False


# Exact correct counts decide improvement; printed percentages are diagnostic only.
@torch.inference_mode()
def evaluate_top1(module, loader):
    module.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            predictions = module(images).argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)

    return correct, total, 100.0 * correct / total

In [ ]:
def cpu_state_dict(module):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in module.state_dict().items()
    }


def load_torch_file(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# Pipeline 5/7 - Save raw best weights and all state needed for exact resume.
def save_checkpoint(state):
    payload = {
        "format_version": 7,
        "epoch": state.epoch,
        "best_validation_correct": state.best_correct,
        "best_validation_top1": state.best_top1,
        "best_model_state_dict": state.best_weights,
        "latest_model_state_dict": cpu_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "stale_epochs": state.stale_epochs,
        "post_regularization_started": (
            state.post_regularization_started
        ),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "loader_rng_state": loader_generator.get_state(),
        "config": {
            "seed": SEED,
            "mixup_stop_epoch": MIXUP_STOP_EPOCH,
            "clean_start_epoch": CLEAN_START_EPOCH,
            "cooldown_lr_factor": COOLDOWN_LR_FACTOR,
        },
    }
    temporary_path = CHECKPOINT_PATH.with_suffix(
        CHECKPOINT_PATH.suffix + ".tmp"
    )
    torch.save(payload, temporary_path)
    temporary_path.replace(CHECKPOINT_PATH)


def load_checkpoint():
    if not RESUME_TRAINING or not CHECKPOINT_PATH.exists():
        return TrainState()

    checkpoint = load_torch_file(CHECKPOINT_PATH)
    if checkpoint.get("format_version") != 7:
        raise ValueError(
            "Checkpoint format mismatch. Use a new CHECKPOINT_PATH "
            "for this hybrid notebook."
        )

    model.load_state_dict(checkpoint["latest_model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    loader_generator.set_state(checkpoint["loader_rng_state"].cpu())
    cuda_states = checkpoint["cuda_rng_states"]
    if torch.cuda.is_available() and cuda_states is not None:
        torch.cuda.set_rng_state_all([state.cpu() for state in cuda_states])

    print(f"Resuming from epoch {checkpoint['epoch'] + 1}.")
    return TrainState(
        epoch=checkpoint["epoch"],
        best_correct=checkpoint["best_validation_correct"],
        best_top1=checkpoint["best_validation_top1"],
        best_weights=checkpoint["best_model_state_dict"],
        stale_epochs=checkpoint["stale_epochs"],
        post_regularization_started=bool(
            checkpoint.get("post_regularization_started", False)
        ),
    )

In [ ]:
# Phase controller - one place owns Mixup, transform, and phase labels.
def training_phase(epoch):
    if epoch < MIXUP_STOP_EPOCH:
        return "mixup"
    if epoch < CLEAN_START_EPOCH:
        return "cooldown"
    return "clean"


def configure_training_phase(epoch):
    phase = training_phase(epoch)
    training_dataset.transform = (
        targeted_train_transform
        if phase != "clean"
        else eval_transform
    )
    return phase


def mixup_loss(images, labels):
    mix_index = torch.randperm(images.size(0), device=images.device)
    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
    lam = max(lam, 1.0 - lam)
    mixed_images = lam * images + (1.0 - lam) * images[mix_index]
    with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
        logits = model(mixed_images)
        return (
            lam * criterion(logits, labels)
            + (1.0 - lam) * criterion(logits, labels[mix_index])
        )


def plain_loss(images, labels):
    with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
        return criterion(model(images), labels)


def train_one_epoch(epoch):
    phase = configure_training_phase(epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    images_seen = 0
    epoch_images = len(training_loader.dataset)
    total_steps = len(training_loader)
    progress = tqdm(
        training_loader,
        desc=f"Epoch {epoch + 1:03d}/{NUM_EPOCHS}",
        leave=False,
    )

    for step, (images, labels) in enumerate(progress, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        use_mixup = (
            phase == "mixup"
            and random.random() < MIXUP_PROBABILITY
        )
        loss = (
            mixup_loss(images, labels)
            if use_mixup
            else plain_loss(images, labels)
        )

        # Correctly normalize the shorter final accumulation group.
        group_start = ((step - 1) // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
        group_size = min(
            ACCUMULATION_STEPS,
            total_steps - group_start,
        )
        scaler.scale(loss / group_size).backward()

        if step % ACCUMULATION_STEPS == 0 or step == total_steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = labels.size(0)
        loss_sum += loss.detach().item() * batch_size
        images_seen += batch_size
        progress.set_postfix(
            phase=phase,
            images=f"{images_seen:,}/{epoch_images:,}",
            loss=f"{loss_sum / images_seen:.4f}",
        )

    return loss_sum / images_seen, phase, images_seen, epoch_images


def should_stop(state):
    return (
        state.epoch >= MIN_EPOCHS
        and state.stale_epochs >= EARLY_STOP_PATIENCE
    )

In [ ]:
# Pipeline 6/7 - Train, validate raw weights, checkpoint, and stop on plateau.
state = load_checkpoint()

for epoch in range(state.epoch, NUM_EPOCHS):
    # Reset once, including when resuming an older post-Mixup checkpoint.
    if (
        epoch >= MIXUP_STOP_EPOCH
        and not state.post_regularization_started
    ):
        state.stale_epochs = 0
        state.post_regularization_started = True
        print("Plateau reset: entering the post-Mixup phase.")

    epoch_lr = optimizer.param_groups[0]["lr"]
    training_loss, phase, images_seen, epoch_images = train_one_epoch(epoch)
    correct, total, top1 = evaluate_top1(model, validation_loader)

    if correct > state.best_correct:
        state.best_correct = correct
        state.best_top1 = top1
        state.best_weights = cpu_state_dict(model)
        state.stale_epochs = 0
    else:
        state.stale_epochs += 1

    state.epoch = epoch + 1
    scheduler.step()
    save_checkpoint(state)

    print(
        f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | "
        f"phase: {phase} | lr: {epoch_lr:.2e} | "
        f"images: {images_seen:,}/{epoch_images:,} | "
        f"train loss: {training_loss:.4f} | "
        f"validation Top-1: {top1:.3f}% ({correct}/{total}) | "
        f"best: {state.best_top1:.3f}% | "
        f"plateau: {state.stale_epochs}/{EARLY_STOP_PATIENCE}"
    )

    # The 99.5% target is reported, never used as an immediate stop condition.
    if should_stop(state):
        print(
            f"Early stop after {state.stale_epochs} epochs "
            "without an exact-count improvement."
        )
        break

## Official test

In [ ]:
# Pipeline 7/7 - Load the best raw checkpoint and evaluate the official test once.
checkpoint = load_torch_file(CHECKPOINT_PATH)
verification_model = build_model()
verification_model.load_state_dict(
    checkpoint["best_model_state_dict"],
    strict=True,
)
test_correct, test_total, test_top1 = evaluate_top1(
    verification_model,
    test_loader,
)

print(
    f"Top-1 accuracy: {test_correct}/{test_total} "
    f"({test_top1:.3f}%)"
)
print(
    f"Target > {TARGET_TOP1:.1f}%: "
    f"{'achieved' if test_top1 > TARGET_TOP1 else 'not achieved'}."
)

try:
    from google.colab import files
except ImportError:
    print("Checkpoint:", CHECKPOINT_PATH.resolve())
else:
    files.download(str(CHECKPOINT_PATH))